# Factor Regression with ETF Proxies

**Quant · Hard · ~25 min · statsmodels + yfinance · needs an internet connection**

Why does a stock go up and down? A factor regression splits its returns into exposures to a few broad drivers (the market, small vs large companies, value vs growth, momentum) plus whatever is left over: alpha and noise. Academic factor data such as Fama-French comes as separate files to download and refresh; here every factor is built from liquid ETFs fetched from Yahoo Finance with `yfinance`, so the notebook needs nothing but an internet connection. The regression is ordinary least squares from `statsmodels` with Newey-West (HAC) standard errors. Nothing is written to disk.

**What you will see**
- The four factor proxies, how each has performed (chart) and how correlated they are (tables)
- A coefficient table: loading, HAC standard error, t-statistic and p-value for every factor, plus R²
- A rolling 126-day market beta, showing how the exposure moves over time
- A plain-English reading of the results

## 1. Why ETF proxies, and which ones

Each style factor is a *long-short* return: one ETF minus another, so the general market move largely cancels and only the tilt remains.

| Factor | Proxy | Reads as |
|---|---|---|
| Market | SPY − T-bill rate | the S&P 500's return above cash |
| Size | IWM − SPY | small caps (Russell 2000) minus large caps |
| Value | IWD − IWF | Russell 1000 Value minus Russell 1000 Growth |
| Momentum | MTUM − SPY | recent winners (MSCI USA Momentum) minus the market |

The risk-free rate is the 13-week US T-bill yield (`^IRX`, also from Yahoo), turned into a daily rate. ETF proxies are long-only and overlap, so the loadings are approximate, but they are free, daily and always up to date.

In [ ]:
# ── Parameters: edit these, then Run All (Ctrl+Shift+Enter) ──────────
STOCK = "AAPL"        # the stock (or fund) to explain
PERIOD = "5y"         # history to download: "3y", "5y", "10y" (MTUM starts in 2013)
ROLL_WINDOW = 126     # rolling-beta window in trading days (about 6 months)
HAC_LAGS = 5          # Newey-West lags: 5 trading days = one week

## 2. Imports and chart style

`statsmodels.api` provides `OLS` and `add_constant`; `RollingOLS` fits the same model on a sliding window in one call.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import yfinance as yf

# One consistent, quiet chart style for every figure in this notebook
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
    "lines.linewidth": 1.6, "legend.frameon": False,
})

## 3. Download prices and the T-bill rate

One download for the stock and the five ETFs, one for the T-bill yield. `^IRX` is quoted in percent per year (4.1 means 4.1%), so dividing by 100 and by 252 gives a daily rate. The T-bill series has a few holidays the stock market does not, so it is forward-filled onto the stock-market dates.

In [ ]:
tickers = list(dict.fromkeys([STOCK, "SPY", "IWM", "IWD", "IWF", "MTUM"]))
data = yf.download(tickers, period=PERIOD, auto_adjust=True, progress=False)
if data.empty:
    raise ValueError(f"No price data for {tickers} - check the symbols or your connection")
closes = data["Close"]
missing = [t for t in tickers if t not in closes or closes[t].isna().all()]
if missing:
    raise ValueError(f"No price data for {missing} - check the symbol or your connection")
returns = closes[tickers].dropna().pct_change().dropna()

tbill = yf.download("^IRX", period=PERIOD, auto_adjust=True, progress=False)
if tbill.empty:
    raise ValueError("No T-bill data (^IRX) - check your connection")
tbill_yield = tbill["Close"].iloc[:, 0]                    # percent per year
rf_daily = (tbill_yield / 100 / 252).reindex(returns.index).ffill().bfill()

print(f"{len(returns)} daily returns, {returns.index[0]:%Y-%m-%d} to {returns.index[-1]:%Y-%m-%d}; "
      f"latest 13-week T-bill yield {tbill_yield.iloc[-1]:.2f}%")

## 4. Build the factor returns

The dependent variable `y` is the stock's **excess return**: its daily return minus the daily T-bill rate. The four factors follow the table above. The chart compounds each factor, so a rising line means the long leg beat the short leg over this period.

Two tables follow: each factor's annualised mean and volatility, and the factor correlations. Strongly correlated factors (above about 0.7 in absolute value) make individual loadings hard to separate.

In [ ]:
y = (returns[STOCK] - rf_daily).rename(f"{STOCK} excess return")
factors = pd.DataFrame({
    "Market": returns["SPY"] - rf_daily,
    "Size": returns["IWM"] - returns["SPY"],
    "Value": returns["IWD"] - returns["IWF"],
    "Momentum": returns["MTUM"] - returns["SPY"],
})

fig, ax = plt.subplots(figsize=(10, 4.5))
for name in factors.columns:
    ax.plot((1 + factors[name]).cumprod(), label=name)
ax.axhline(1, color="#898781", linewidth=0.8, linestyle="--")
ax.set_title("Cumulative factor returns (growth of $1 in each proxy)")
ax.set_xlabel("Date")
ax.set_ylabel("Growth of $1")
ax.legend(loc="upper left")
plt.tight_layout()

display(pd.DataFrame({"Annual mean (%)": factors.mean() * 252 * 100,
                      "Annual volatility (%)": factors.std() * np.sqrt(252) * 100}).round(2))
factors.corr().round(2)

## 5. The regression

We fit

`excess return = alpha + b_mkt × Market + b_size × Size + b_value × Value + b_mom × Momentum + error`

by ordinary least squares. Daily returns have volatility clusters and a little autocorrelation, which makes ordinary standard errors too optimistic. **Newey-West (HAC) standard errors** correct for both, so the t-statistics and p-values below are the honest ones; in statsmodels it is just `cov_type="HAC"`. Alpha is a daily return, so the table shows it (and its standard error) scaled to percent per year; scaling does not change its t-statistic or p-value.

In [ ]:
X = sm.add_constant(factors).rename(columns={"const": "Alpha"})
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": HAC_LAGS})

coefs = pd.DataFrame({
    "Coefficient": model.params,
    "Std error (HAC)": model.bse,
    "t-stat": model.tvalues,
    "p-value": model.pvalues,
})
coefs.loc["Alpha", ["Coefficient", "Std error (HAC)"]] *= 252 * 100   # daily -> % per year
coefs = coefs.rename(index={"Alpha": "Alpha (% per year)"})
coefs["Significant at 5%"] = np.where(coefs["p-value"] < 0.05, "yes", "no")
coefs.index.name = "Term"

print(f"R² = {model.rsquared:.3f}   adjusted R² = {model.rsquared_adj:.3f}   "
      f"observations = {int(model.nobs)}   HAC lags = {HAC_LAGS}")
coefs.round(3)

## 6. Rolling market beta

One regression gives one average answer for the whole period. Fitting it again on a sliding 126-day window (about six months) shows how the exposure has moved. `RollingOLS` fits every window in one go. The first line is the market beta from the full four-factor model, the second the simple one-factor (CAPM) beta for comparison, and the dashed line the full-period estimate from the regression above.

In [ ]:
rolling_4f = RollingOLS(y, X, window=ROLL_WINDOW).fit()
rolling_capm = RollingOLS(y, X[["Alpha", "Market"]], window=ROLL_WINDOW).fit()
betas = pd.DataFrame({
    "Market beta, 4-factor model": rolling_4f.params["Market"],
    "Market beta, market only (CAPM)": rolling_capm.params["Market"],
}).dropna()

fig, ax = plt.subplots(figsize=(10, 4))
for name in betas.columns:
    ax.plot(betas[name], label=name)
ax.axhline(model.params["Market"], color="#898781", linestyle="--", linewidth=1,
           label=f"Full-period beta ({model.params['Market']:.2f})")
ax.set_title(f"{STOCK}: rolling {ROLL_WINDOW}-day market beta")
ax.set_xlabel("Date (end of window)")
ax.set_ylabel("Beta")
ax.legend(loc="best")
plt.tight_layout()

b4 = betas.iloc[:, 0]
print(f"4-factor market beta ranged from {b4.min():.2f} to {b4.max():.2f}; latest {b4.iloc[-1]:.2f}")

## 7. How to read the results

- **Market beta**: how much the stock moves for a 1% market move above cash. Above 1 amplifies the market, below 1 dampens it.
- **Size**: positive means it behaves like small caps; negative like a large cap (typical for mega caps such as AAPL).
- **Value**: positive means a value tilt (banks, energy); negative a growth tilt (most of tech).
- **Momentum**: positive means it tends to move with recent winners.
- **Alpha**: the return left after accounting for the factors. A t-statistic beyond about ±2 (p-value below 0.05) is the usual bar for "probably not luck"; most alphas fail it.
- **R²**: the share of day-to-day variance the factors explain. The rest is company-specific news.

Caveats: the proxies overlap (IWM and MTUM also carry market exposure, which is why SPY is subtracted from them); results depend on the period, so try `PERIOD = "10y"` and compare; and a regression explains the past, it does not forecast. The next cell turns the fitted numbers into sentences for the stock you chose.

In [ ]:
b, p, t = model.params, model.pvalues, model.tvalues
print(f"{STOCK} moves about {b['Market']:.2f}x the market's excess return (t = {t['Market']:.1f}).")

words = {"Size": ("small caps", "large caps"),
         "Value": ("value stocks", "growth stocks"),
         "Momentum": ("recent winners", "recent laggards")}
for f, (pos, neg) in words.items():
    if p[f] < 0.05:
        print(f"- {f}: loading {b[f]:+.2f} (p = {p[f]:.3f}), so it behaves like {pos if b[f] > 0 else neg}.")
    else:
        print(f"- {f}: loading {b[f]:+.2f} is not statistically significant (p = {p[f]:.2f}).")

alpha_year = b["Alpha"] * 252 * 100
verdict = "statistically significant" if p["Alpha"] < 0.05 else "not significant, so it could be luck"
print(f"Alpha: {alpha_year:+.1f}% per year after the factors, {verdict} (p = {p['Alpha']:.2f}).")
print(f"The factors explain {model.rsquared:.0%} of the daily variance; the rest is specific to {STOCK}.")

## Try next

- Explain other names: `XOM` (value), `TSLA` (high beta), `JNJ` (defensive), or a fund such as `ARKK`.
- Use weekly returns to reduce noise: resample `closes` with `.resample("W-FRI").last()` before `pct_change()`, divide the T-bill yield by 52 instead of 252, and lower `ROLL_WINDOW` to about 26 weeks.
- Add a quality proxy (`QUAL − SPY`) or a low-volatility proxy (`USMV − SPY`) as a fifth factor.

---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*